# Lab 10 — Part B: Change Data Capture
Delta Change Data Feed as the CDC source. Changes are captured incrementally
and applied to an SCD Type 2 target with MERGE INTO.

Flow: Neon `sales.customers` → (initial load via federation) → `customers_src` (Delta, CDF on)
→ `table_changes()` → `dim_customers_scd2`

In [0]:
USE CATALOG dbr_dev_ua5816bd;
USE SCHEMA natalia_lab10;
SET spark.databricks.execution.timeout = 3600;

## Reset (demo only)

In [0]:
DROP TABLE IF EXISTS dim_customers_scd2;
DROP TABLE IF EXISTS cdc_checkpoint;
DROP TABLE IF EXISTS customers_src;

In [0]:
CREATE TABLE customers_src
TBLPROPERTIES (delta.enableChangeDataFeed = true)
AS
SELECT customer_id, full_name, email, city, segment, updated_at
FROM natalia_neon_catalog.sales.customers;

In [0]:
DESCRIBE HISTORY customers_src;

In [0]:
SELECT customer_id, city, segment, _change_type, _commit_version, _commit_timestamp
FROM table_changes('customers_src', 0)
ORDER BY customer_id
LIMIT 10;

## Target: SCD Type 2 table + processing checkpoint

In [0]:
CREATE TABLE IF NOT EXISTS dim_customers_scd2 (
  customer_id INT NOT NULL COMMENT 'Business key from the source',
  full_name STRING,
  email STRING,
  city STRING,
  segment STRING,
  valid_from TIMESTAMP NOT NULL COMMENT 'When this version became effective (source commit time)',
  valid_to TIMESTAMP COMMENT 'When this version was superseded; NULL for the current version',
  is_current BOOLEAN NOT NULL COMMENT 'True for the latest version of the customer',
  is_deleted BOOLEAN NOT NULL COMMENT 'True if the customer was deleted in the source',
  _source_version BIGINT NOT NULL COMMENT 'customers_src Delta version that produced this row'
)
COMMENT 'Customer dimension, SCD Type 2, built incrementally from customers_src Change Data Feed';

In [0]:
CREATE TABLE IF NOT EXISTS cdc_checkpoint (
  source_table STRING NOT NULL,
  last_processed_version  BIGINT NOT NULL
  );

In [0]:
INSERT INTO dim_customers_scd2
SELECT
  customer_id, full_name, email, city, segment,
  _commit_timestamp AS valid_from,
  NULL AS valid_to,
  true AS is_current,
  false AS is_deleted,
  _commit_version AS _source_version
FROM table_changes('customers_src', 0, 0)
WHERE _change_type = 'insert';

In [0]:
INSERT INTO cdc_checkpoint VALUES ('customers_src', 0);

In [0]:
SELECT count(*) AS total_rows, sum(CAST(is_current AS INT)) AS current_rows
FROM dim_customers_scd2;

## Incremental CDC pipeline: `run_cdc_batch()`
One reusable function, called after every round of source changes:
1. read the checkpoint (last processed version) and the latest source version;
2. read only the new changes via `table_changes()` (empty batch if there are none);
3. keep one event per customer (the latest `_commit_version`), drop `update_preimage`;
4. apply the batch to the SCD2 target with MERGE INTO;
5. move the checkpoint to the latest processed version.

Running it again without new source changes is a no-op (idempotent).

In [0]:
%python
def run_cdc_batch(source="customers_src", target="dim_customers_scd2", checkpoint="cdc_checkpoint"):
    last_processed = spark.sql(
        f"SELECT last_processed_version FROM {checkpoint} WHERE source_table = '{source}'"
    ).first()[0]
    latest_version = spark.sql(f"DESCRIBE HISTORY {source} LIMIT 1").first()["version"]

    if latest_version <= last_processed:
        return

    spark.sql(
        f"SELECT * FROM table_changes('{source}', {last_processed + 1}, {latest_version})"
    ).createOrReplaceTempView("cdc_batch_raw")

    spark.sql("""
        CREATE OR REPLACE TEMP VIEW cdc_batch AS
        SELECT * EXCEPT (rn)
        FROM (
          SELECT
            *,
            row_number() OVER (PARTITION BY customer_id ORDER BY _commit_version DESC) AS rn
          FROM cdc_batch_raw
          WHERE _change_type IN ('insert', 'update_postimage', 'delete')
        )
        WHERE rn = 1
    """)

    spark.sql(f"""
        MERGE INTO {target} AS t
        USING (
          SELECT customer_id AS merge_key, *
          FROM cdc_batch

          UNION ALL

          SELECT CAST(NULL AS INT) AS merge_key, *
          FROM cdc_batch
          WHERE _change_type IN ('insert', 'update_postimage')
        ) AS s
        ON t.customer_id = s.merge_key AND t.is_current = true

        WHEN MATCHED AND s._change_type IN ('insert', 'update_postimage') THEN
          UPDATE SET
            t.valid_to   = s._commit_timestamp,
            t.is_current = false

        WHEN MATCHED AND s._change_type = 'delete' THEN
          UPDATE SET
            t.valid_to   = s._commit_timestamp,
            t.is_current = false,
            t.is_deleted = true

        WHEN NOT MATCHED AND s.merge_key IS NULL THEN
          INSERT (customer_id, full_name, email, city, segment,
                  valid_from, valid_to, is_current, is_deleted, _source_version)
          VALUES (s.customer_id, s.full_name, s.email, s.city, s.segment,
                  s._commit_timestamp, NULL, true, false, s._commit_version)
    """)

    spark.sql(f"""
        UPDATE {checkpoint}
        SET last_processed_version = {latest_version}
        WHERE source_table = '{source}'
    """)

## simulated source changes
Each statement is a separate Delta commit (new version):
- v1: customer 7 moves Lutsk → Kyiv (UPDATE)
- v2: customer 15 upgraded regular → gold (UPDATE)
- v3: customer 42 deleted (DELETE)
- v4: new customer 1001 (INSERT)

In [0]:
UPDATE customers_src
SET city = 'Kyiv', updated_at = current_timestamp()
WHERE customer_id = 7;

In [0]:
UPDATE customers_src
SET segment = 'gold', updated_at = current_timestamp()
WHERE customer_id = 15;

In [0]:
DELETE FROM customers_src
WHERE customer_id = 42;

In [0]:
INSERT INTO customers_src (customer_id, full_name, email, city, segment, updated_at)
VALUES (1001, 'Customer 1001', 'customer1001@example.com', 'Lviv', 'silver', current_timestamp());

In [0]:
DESCRIBE HISTORY customers_src;

In [0]:
SELECT customer_id, city, segment, _change_type, _commit_version, _commit_timestamp
FROM table_changes('customers_src', 1)
ORDER BY _commit_version, customer_id, _change_type;

In [0]:
%python
run_cdc_batch()

In [0]:
SELECT customer_id, city, segment, _change_type, _commit_version
FROM cdc_batch
ORDER BY customer_id;

In [0]:
SELECT customer_id, city, segment, valid_from, valid_to, is_current, is_deleted, _source_version
FROM dim_customers_scd2
WHERE customer_id IN (7, 15, 42, 1001)
ORDER BY customer_id, valid_from;

In [0]:
SELECT
  count(*) AS total_versions,
  count_if(is_current) AS current_customers,
  count_if(is_deleted) AS deleted_customers,
  (SELECT count(*) FROM customers_src) AS customers_in_source
FROM dim_customers_scd2;

In [0]:
SELECT * FROM cdc_checkpoint;

## harder cases
- v5: customer 7 moves Kyiv → Odesa
- v6: customer 7 moves again Odesa → Lviv (two changes of one key before the next pipeline run)
- v7: customer 42 comes back (insert of a key that already has closed, deleted history)
- v8–v9: customer 100 is deleted and re-created within the same batch
  (after dedup only the insert is left, while the target still has a current version)

In [0]:
UPDATE customers_src
SET city = 'Odesa', updated_at = current_timestamp()
WHERE customer_id = 7;

In [0]:
UPDATE customers_src
SET city = 'Lviv', updated_at = current_timestamp()
WHERE customer_id = 7;

In [0]:
INSERT INTO customers_src (customer_id, full_name, email, city, segment, updated_at)
VALUES (42, 'Customer 42', 'customer42@example.com', 'Kharkiv', 'regular', current_timestamp());

In [0]:
DELETE FROM customers_src
WHERE customer_id = 100;

In [0]:
INSERT INTO customers_src (customer_id, full_name, email, city, segment, updated_at)
VALUES (100, 'Customer 100', 'customer100@example.com', 'Odesa', 'silver', current_timestamp());

In [0]:
%python
run_cdc_batch()

In [0]:
SELECT customer_id, city, segment, _change_type, _commit_version
FROM cdc_batch
ORDER BY customer_id;

In [0]:
SELECT customer_id, city, segment, valid_from, valid_to, is_current, is_deleted, _source_version
FROM dim_customers_scd2
WHERE customer_id IN (7, 42, 100)
ORDER BY customer_id, valid_from;

In [0]:
SELECT
  count(*) AS total_versions,
  count_if(is_current) AS current_customers,
  count_if(is_deleted) AS deleted_customers,
  (SELECT count(*) FROM customers_src) AS customers_in_source
FROM dim_customers_scd2;

In [0]:
%python
run_cdc_batch()

In [0]:
SELECT * FROM cdc_checkpoint;

In [0]:
DESCRIBE HISTORY dim_customers_scd2 LIMIT 3;